# Spatial Correctness and Elevation vs Bird Richness

This notebook analyzes bird richness with elevation_meters after spatial thinning.

Scope:
- Spatial thinning: one record per (district, 5 km cell, species)
- Cell-level bird richness
- Elevation-richness relationships at cell and district levels


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.formula.api as smf

sns.set_theme(style="whitegrid", context="paper")
pd.set_option("display.max_columns", 80)
plt.rcParams["figure.dpi"] = 120
RANDOM_SEED = 42


## 1. Load bird + elevation columns

In [ ]:
candidate_paths = [
    Path("file6.csv"),
    Path("../file6.csv"),
    Path("../../file6.csv"),
    Path("final5.csv"),
    Path("../final5.csv"),
    Path("../../final5.csv"),
]

data_path = next((p for p in candidate_paths if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("Could not find file6.csv or final5.csv from this notebook location.")

required_cols = [
    "stateProvince",
    "verbatimScientificName",
    "decimalLatitude",
    "decimalLongitude",
    "eventDate",
    "elevation_meters",
]

raw = pd.read_csv(data_path, low_memory=False)
missing = [c for c in required_cols if c not in raw.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = raw[required_cols].copy()
df["eventDate"] = pd.to_datetime(df["eventDate"], errors="coerce")
for c in ["decimalLatitude", "decimalLongitude", "elevation_meters"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df["stateProvince"] = df["stateProvince"].astype(str).str.strip()
df["verbatimScientificName"] = df["verbatimScientificName"].astype(str).str.strip()

df = df.dropna(subset=["stateProvince", "verbatimScientificName", "decimalLatitude", "decimalLongitude", "eventDate", "elevation_meters"])

print(f"Using: {data_path.resolve()}")
print(f"Rows after cleaning: {len(df):,}")
print(f"Districts: {df['stateProvince'].nunique():,}")
print(f"Species: {df['verbatimScientificName'].nunique():,}")


## 2. Spatial thinning (district, 5 km cell, species)

In [ ]:
def add_5km_grid(frame: pd.DataFrame) -> pd.DataFrame:
    out = frame.copy()
    lon0 = float(out["decimalLongitude"].median())
    lat0 = float(out["decimalLatitude"].median())
    lat_rad = np.radians(out["decimalLatitude"].to_numpy())
    out["x_km"] = (out["decimalLongitude"].to_numpy() - lon0) * 111.320 * np.cos(lat_rad)
    out["y_km"] = (out["decimalLatitude"].to_numpy() - lat0) * 110.574
    out["grid_x"] = np.floor(out["x_km"] / 5.0).astype(int)
    out["grid_y"] = np.floor(out["y_km"] / 5.0).astype(int)
    out["cell_id"] = (
        out["stateProvince"].astype(str)
        + "|"
        + out["grid_x"].astype(str)
        + "_"
        + out["grid_y"].astype(str)
    )
    return out

def thin_per_species_cell(frame: pd.DataFrame, seed: int = 42) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    work = frame.copy()
    work["_thin_key"] = (
        work["stateProvince"].astype(str)
        + "|"
        + work["grid_x"].astype(str)
        + "_"
        + work["grid_y"].astype(str)
        + "|"
        + work["verbatimScientificName"].astype(str)
    )
    work["_rand"] = rng.random(len(work))
    return (
        work.sort_values("_rand")
        .groupby("_thin_key", as_index=False)
        .head(1)
        .drop(columns=["_thin_key", "_rand"])
    )

df_grid = add_5km_grid(df)
df_thin = thin_per_species_cell(df_grid, seed=RANDOM_SEED)

print(f"Records before thinning: {len(df_grid):,}")
print(f"Records after thinning:  {len(df_thin):,}")
print(f"Retained fraction:       {len(df_thin)/len(df_grid):.2%}")


## 3. Cell-level richness + elevation

In [ ]:
cell_summary = (
    df_thin.groupby(["stateProvince", "cell_id"], as_index=False)
    .agg(
        species_richness=("verbatimScientificName", "nunique"),
        elevation_mean=("elevation_meters", "mean"),
    )
)

print(f"Cell rows: {len(cell_summary):,}")
cell_summary.head(10)


## 4. Richness vs elevation (cell and district)

In [ ]:
r_cell, p_cell = stats.spearmanr(cell_summary["species_richness"], cell_summary["elevation_mean"], nan_policy="omit")
print(f"Cell-level Spearman (richness vs elevation): r={r_cell:.3f}, p={p_cell:.3g}")

district_summary = (
    cell_summary.groupby("stateProvince", as_index=False)
    .agg(
        mean_cell_richness=("species_richness", "mean"),
        mean_elevation=("elevation_mean", "mean"),
        n_cells=("cell_id", "count"),
    )
)

r_dist, p_dist = stats.spearmanr(district_summary["mean_cell_richness"], district_summary["mean_elevation"], nan_policy="omit")
print(f"District-level Spearman (richness vs elevation): r={r_dist:.3f}, p={p_dist:.3g}")
district_summary.sort_values("mean_cell_richness", ascending=False).head(30)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

sns.regplot(
    data=cell_summary,
    x="elevation_mean",
    y="species_richness",
    scatter_kws={"s": 14, "alpha": 0.12, "color": "#2c3e50"},
    line_kws={"color": "#c0392b"},
    ax=axes[0],
)
axes[0].set_title("Cell richness vs elevation")
axes[0].set_xlabel("Cell mean elevation (m)")
axes[0].set_ylabel("Species richness")

sns.scatterplot(
    data=district_summary,
    x="mean_elevation",
    y="mean_cell_richness",
    size="n_cells",
    sizes=(45, 380),
    color="#1f618d",
    alpha=0.85,
    ax=axes[1],
)
axes[1].set_title("District mean richness vs mean elevation")
axes[1].set_xlabel("District mean elevation (m)")
axes[1].set_ylabel("Mean cell richness")

plt.tight_layout()
plt.show()


## 5. Elevation model and profile

In [ ]:
model_df = cell_summary.copy()
model_df["z_elevation"] = (model_df["elevation_mean"] - model_df["elevation_mean"].mean()) / model_df["elevation_mean"].std(ddof=0)
fit = smf.ols("species_richness ~ z_elevation", data=model_df).fit()
print(fit.summary().tables[0])
print(fit.summary().tables[1])

district_summary = district_summary.sort_values("mean_elevation", ascending=False)
plt.figure(figsize=(9.6, max(5, 0.28 * len(district_summary))))
plt.barh(district_summary["stateProvince"], district_summary["mean_elevation"], color="#7d3c98")
plt.title("District mean elevation")
plt.xlabel("Elevation (m)")
plt.tight_layout()
plt.show()


## 6. Optional exports

In [ ]:
# Uncomment to export outputs.
# cell_summary.to_csv("elevation_cell_summary.csv", index=False)
# district_summary.to_csv("elevation_district_summary.csv", index=False)
